In [8]:
# @title 0. Import necessary libraries

import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [9]:
# @title 1. Import data and Split data

df = pd.read_csv("vibration_features.csv")
X_raw = df.select_dtypes(
    include=["float32", "float64", "int64"]
).values.astype("float32")

# Train / Validation split
X_train_raw, X_val_raw = train_test_split(
    X_raw, test_size=0.20, random_state=42, shuffle=True
)

print(f"  • Training samples   : {X_train_raw.shape[0]}")
print(f"  • Validation samples : {X_val_raw.shape[0]}")
print(f"  • Feature dimension  : {X_train_raw.shape[1]}")

FileNotFoundError: [Errno 2] No such file or directory: 'vibration_features.csv'

In [10]:
# @title 2. Preprocessing & Normalization:

# Perform normalization for training and validation sets:
# -> .fit() -> Calculate and Store statistical parameters from datasets
# -> .transform() -> Perform transformation formula based on .fit()'s parameters
# -> .fit_transform() -> Combine .fit() and .transform() 
# -> Since .fit() saves parameters directly to instance attributes on RAM, validation and testing datasets mustn't use .fit()
# -> to avoid Data Leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)

# Export parameters to hardcode in ESP32 firmware for edge inference later
# -> The data must be float32 for being suitable for ESP32; it needs to be hardcode as contants for quicker access.
scaler_mean = scaler.mean_.astype(np.float32)
scaler_std = scaler.scale_.astype(np.float32)

print(f"Normalized train shape: {X_train.shape}")
print(f"Normalized val shape  : {X_val.shape}")
print(f"\nMean per feature (for C++ header):\n{scaler_mean.tolist()}")
print(f"\nStd per feature (for C++ header):\n{scaler_std.tolist()}")

NameError: name 'X_train_raw' is not defined

In [ ]:
# @title 3. Model Architecture:

# PyTorch Data Loaders:
train_tensor = torch.tensor(X_train, dtype=torch.float32)
val_tensor = torch.tensor(X_val, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(train_tensor, train_tensor), batch_size=32, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(val_tensor, val_tensor), batch_size=32, shuffle=False
)

# Define model:
class VibrationAutoencoder(nn.Module):

  def __init__(self, input_dim: int):
    super().__init__()
    # Encoder's Architecture: Input -> 16 -> 4
    self.encoder = nn.Sequential(
        nn.Linear(input_dim, 16),
        nn.ReLU(),
        nn.Linear(16, 4),
        nn.ReLU(),
    )
    # Decoder's Architecture: 4 -> 16 -> The sketch to try to be similar with Input
    self.decoder = nn.Sequential(
        nn.Linear(4, 16),
        nn.ReLU(),
        nn.Linear(16, input_dim),  # Linear Activation in output
    )

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.decoder(self.encoder(x))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VibrationAutoencoder(input_dim=X_train.shape[1]).to(device)

criterion_mse = nn.MSELoss()
criterion_mae = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# @title 4. Training Loop with Early Stopping:
epochs = 100
patience = 10
best_val_loss = float("inf")
patience_counter = 0
best_model_weights = copy.deepcopy(model.state_dict())

history = {"loss": [], "mae": [], "val_loss": [], "val_mae": []}

for epoch in range(epochs):
  # --- Training Phase ---
  model.train()
  train_mse, train_mae = 0.0, 0.0
  for x_batch, _ in train_loader:
    x_batch = x_batch.to(device)

    optimizer.zero_grad()
    reconstruction = model(x_batch)
    loss = criterion_mse(reconstruction, x_batch)
    loss.backward()
    optimizer.step()

    train_mse += loss.item() * len(x_batch)
    train_mae += criterion_mae(reconstruction, x_batch).item() * len(x_batch)

  # --- Validation Phase ---
  model.eval()
  val_mse, val_mae = 0.0, 0.0
  with torch.no_grad():
    for x_val_batch, _ in val_loader:
      x_val_batch = x_val_batch.to(device)
      val_recon = model(x_val_batch)

      val_mse += criterion_mse(val_recon, x_val_batch).item() * len(x_val_batch)
      val_mae += criterion_mae(val_recon, x_val_batch).item() * len(x_val_batch)

  # Average epoch metrics
  epoch_train_loss = train_mse / len(train_tensor)
  epoch_train_mae = train_mae / len(train_tensor)
  epoch_val_loss = val_mse / len(val_tensor)
  epoch_val_mae = val_mae / len(val_tensor)

  history["loss"].append(epoch_train_loss)
  history["mae"].append(epoch_train_mae)
  history["val_loss"].append(epoch_val_loss)
  history["val_mae"].append(epoch_val_mae)

  # Early Stopping Check
  if epoch_val_loss < best_val_loss:
    best_val_loss = epoch_val_loss
    patience_counter = 0
    best_model_weights = copy.deepcopy(model.state_dict())
  else:
    patience_counter += 1
    if patience_counter >= patience:
      print(f"Early stopping triggered at epoch {epoch + 1}")
      break

# Restore best weights and save metrics
model.load_state_dict(best_model_weights)
metrics_df = pd.DataFrame(history)
metrics_df.to_csv("training_metrics.csv", index_label="epoch")

In [ ]:
# @title 4. See the linear graphs of Training and Validation accuracy:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), dpi=100)

axes[0].plot(metrics_df["loss"], label="Train MSE")
axes[0].plot(metrics_df["val_loss"], label="Val MSE", linestyle="--")
axes[0].set_title("Reconstruction Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[0].grid(True, linestyle=":")
axes[0].legend()

axes[1].plot(metrics_df["mae"], label="Train MAE")
axes[1].plot(metrics_df["val_mae"], label="Val MAE", linestyle="--")
axes[1].set_title("Mean Absolute Error (MAE)")
axes[1].set_xlabel("Epoch")
axes[1].grid(True, linestyle=":")
axes[1].legend()

plt.tight_layout()
plt.show()